# Assignment 09: Jazz Solo Improvisation using Generative Neural Networks
---

**Due Date:** Wednesday 07/01/2026 (by midnight)

**Please fill these in before submitting, just in case I accidentally mix up file names while grading**:

Name: Jane Hacker

CWID-5: (Last 5 digits of cwid)

# Introduction 

Welcome to our programming assignment on doing generative AI using neural networks.  In this
notebook, you will implement a model that uses recurrent and/or Transformer layers to 
learn sequence-to-sequence next token prediction so that you can make a Jazz Solo music generator.  At the end,
you'll be able to listen to the Jazz solos that your networks generates.

![Jazz Music Generation](../figures/jazz.jpg)
 
**Instructions:**

- As with the previous assignment, you will need to create the function declarations asked for
  in `src/assg_tasks.py`.  Make sure you use
  [Python Docstrings](https://www.geeksforgeeks.org/python-docstrings/) and are generally
  following [Pep8 Python Style Guide](https://peps.python.org/pep-0008/) for your code.
- Cells with `### TESTED` comment contain unit tests that are run on your implementation.  You will
  need to uncomment the call to the unit tests, but otherwise need to stay as given in the original
  notebook.
- Likewise since you need to write your declaration of the functions asked for the tasks, don't forget
  to uncomment/add the appropriate `from assg_src include X` statements in both this notebook and
  in the `../src/test_assg_tasks.py`

**In this assignment, you will:**

- Create a sequence-to-sequence next token predictor.
- Develop the code to use a trained next token predictor to generate music
  using a stochastic sampling strategy.
- Generate your own jazz music with deep learning.
- Practice using the flexible Keras Functional API to create networks

# Packages

The following imports should be all of the packages that you will need for this assignment.
We are using the Keras API in this assignment and in future assignments, so the `tensorflow` and `keras` modules
you need are now available in the notebook.

In [ ]:
# assignment wide imports go here, usually all of your imports for notebooks should
# be put up at the top here, if they were not given to you at the start of the assignment
import numpy as np
import matplotlib.pyplot as plt
import IPython
import random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.utils import to_categorical

In [ ]:
# The following ipython magic will reload changed file/modules.
# So when editing function in source code modules, you should
# be able to just rerun the cell, not restart the whole kernel.
%load_ext autoreload
%autoreload 3

The imports of the function you will write have been commented out here this time.  You will need to uncomment
the imports once you declare and write your functions here, and also in the `src/test_assg_tasks.py` file to
run the unit tests on your work.

In [ ]:
# assignment function imports for doctests and github autograding
# these are required for assignment autograding
# NOTE: the following requires that your PYTHONPATH is correctly setup to include the src directory in your
#   your jupyter lab environment
from assg_utils import run_unittests, run_doctests
from assg_tasks import reweight_distribution
from assg_tasks import plot_history
from music_utils import load_music_dataset
from music_utils import generate_music
from music_utils import mid2wav

# Corpus of Jazz Music

You would like to create a jazz music piece specially for a friend's birthday. However,
you don't know how to play any instruments, or how to compose music. Fortunately, you
know deep learning and will solve this problem using an LSTM network! 

You will train a network to generate novel jazz solos in a style representative of a body
of performed work. 😎🎷


## Dataset

To get started, you'll train your algorithm on a corpus of Jazz music. Run the cell below
to listen to a snippet of the audio from the training set:

In [ ]:
IPython.display.Audio('../data/30s_seq.wav')

The preprocessing of the musical data has been taken care of already, which for this notebook
means it's been rendered in terms of musical "values." 

## What are musical "Values"?

You can informally think of each "value" as a note, which comprises a pitch and duration.
For example, if you press down a specific piano key for 0.5 seconds, then you have just
played a note. In music theory, a "value" is actually more complicated than this, 
specifically, it also captures the information needed to play multiple notes at the
same time. For example, when playing a music piece, you might press down two piano
keys at the same time (playing multiple notes at the same time generates what's called
a "chord"). But you don't need to worry about the details of music theory for this assignment. 

## Music as a sequence of values

- For the purposes of this assignment, all you need to know is that you'll obtain a dataset
  of values, and will use an RNN model to generate sequences of values. 
- Your music generation system will use 90 unique values (this is the vocabulary, and your vocabulary size is 90).

Run the following code to load the raw music data and preprocess it into values.
This might take a few minutes!

In [ ]:
X, Y, vocab_size, vocab_dict, chords = load_music_dataset('../data/original_metheny.mid')
num_samples = X.shape[0]
sequence_length = X.shape[1]
print('number of training examples:', num_samples)
print('sequence_length:', sequence_length)
print('vocabulary size (number of music "values":', vocab_size)
print('shape of X:', X.shape)
print('Shape of Y:', Y.shape)
print('Number of chords', len(chords))

You have just loaded the following:

- `X`, `Y` These are a `(num_samples, sequence_length, vocab_size)` shaped tensor to be used for training inputs and training targets respectively.
  - You have 60 training examples `num_samples = 60`, each of which is a snippet of `sequence_length = 30` musical values.
  - At each time step, the input is one of 90 different possible musical values (the `vocab_size`)
    encoded as a one-hot vector.
    - For example `X[i, t, :]` is a one-hot vector representing values of the i-th example at time t.
  - `Y` is essentially the same as `X`, but shifted one time step, to be used as target
    of sequence-to-sequence next-token training.
  - The first item in `X` is the start token 0, so when generating we can prompt with 0 by default to begin a sequence.
- `vocab_dict` is a python dictionary of `vocab_size` `(value_index, musical value)` key value
  pairs, mapping the vocabulary index integers 0 through 89 to musical values.
- `chords` are chords used in the input midi, these will be used to play the solo once you generate it.

# Task 1: Build a Sequence-to-Sequence Prediction Model using GRU Layer

You will start by creating a model to do N+1 sequence-to-sequence prediction.  Lets first use a Bidirectional layer with
a pair of GRU layers in it.  Set the size of your hidden dimensions in your GRU layers to 32.

In your first task, you will create a function to return a model for sequence-to-sequence
next token prediction.  For your first model let's start with a GRU sequence layer.

**Task**: Declare and implement a function named `gru_music_generator_model()`.  This model should take
the following inputs:

- `vocab_size` The size of the vocabulary being used in the one-hot encoded input and target sequences.
- `num_dim` The number of hidden dimensions to use for the GRU layer.

You should use the Keras functional API to create a model and return it.  Your model is can be pretty simple,
you only need the inputs, a GRU layer, and a Dense layer.  Make sure that your GRU layer is returning all of the
sequences (not just the final sequence prediction).  Your output layer should be a Dense layer
that is using `softmax` activation to give a probability distribution of which of the  `vocab_size` tokens is most likely to be the next token
in the sequence.

The tests for this function are only checking that you use a GRU layer and are using the number of hidden
dimensions, and that your last Dense layer looks of the correct shape and is using the correct activation for
this task.  Use 32 as the number of dimensions for the hidden representations of your GRU layer.

In [ ]:
### TESTED function gru_music_generator_model()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_gru_music_generator_model'])

num_dim = 32
#model = gru_music_generator_model(vocab_size, num_dim)
#model.summary()
#keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

**Expected Output**:  The model summary and network graph should look like the following:

```
Model: "gru_music_generator_model"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, None, 123)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, None, 32)       │        15,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ outputs (Dense)                 │ (None, None, 123)      │         4,059 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 19,131 (74.73 KB)
 Trainable params: 19,131 (74.73 KB)
 Non-trainable params: 0 (0.00 B)
```

![Expected GRU Model Graph](../figures/expected_gru_music_generator_model.png)

**Task**: In the following cell(s) train your model on the input and offset output music sequences.  Use the following guidance

- Use a basic `rmsprop` optimizer and report `accuracy` metric when fitting the model.  Use the correct loss for doing
  a multiclass classification.  In this case the X inputs and the Y output targets both have one-hot encoded representations.
- Train your model for 200 epochs (it will be fast).  Use a batch size of 10.  You don't need any validation data here for training a model for generating music.
- Plot the learning curve history, `plot_history()` is available to use.

In [ ]:
# compile your model for multi class classification using rmsprop and reporting on accuracy

In [ ]:
# fit your model for 200 epochs with a batch size of 10

In [ ]:
# show the learning curves of the training loss and accuracy


**Expected Results** The task here is relatively small.  If all is working you should see that after 200 epochs the training accuracy will usually look like
it is leveling off around 92%.  If you train more epochs it would probably continue to increase somewhat.  

As we discussed in our class materials, the next token prediction accuracy here is not really all that meaningful in the context of building a system
to generate musical solos.  But you should see the loss decrease and that accuracy does improve above 90%, which means that the model is
learning something here.

But to evaluate what it is doing, we will have to generate some solos from the model and listen to them.

# Task 2: Generate Music using Stochastic Sampling Strategy

For a generative AI system, how a model is doing is a bit more subjective than a simple accuracy or MAE number.  What does 92% next token prediction accuracy mean
in terms of generating interesting musical solos?

**Task** Declare and implement a function named `generate_jazz_solo()`. This function is similar in some ways to the text-generation callback
(Textbook section 12.1.5).  In order to generate solos using your trained model, we need to:
1. start with a prompt as the first musical value for the solo
2. ask the model to predict the next musical value after the current sequence generated so far (returns a `softmax` probability distribution).

   **Note** the model returns the next prediction for each step in current sequence, so for example at the 5th iteration if you give it an input
   sequence with 5 items in the sequence, it returns 5 predictions.  So you need the last prediction returned in the sequence as the one you want
   to reweight and sample from to generate the next musical value.

   **Note**: Please set `verbose=False` when using your model's `predict()` here so that we don't get a bunch of output when calling.
   
4. Use the `reweight_distributions()` (copied into `assg_tasks.py` from our text for you) and the `temperature` parameter to reweight the probability distribution
5. As our text did, use the `np.random.multinomial()` to randomly sample which musical value token to perform next.  The `multinomial()` function takes the
   (reweighted) probability distribution, will select one of the `vocab_size` tokens based on this distribution, then return a one-hot representation of which token index was selected.
6. Append the one-hot selected next musical value to end of current solo / sentence
7. Go back to 2 and keep looping for the desired sequence length to generate

It is suggested, since the model's `predict()` will return a one-hot `(vocab_size,)` vector, and you need to pass in one-hot encoded tokens into predict, that you start with
a NumPy array of shape `(1, 1, vocab_size)` as your solo / sentence.  The 1 item in the batch has currently 1 musical value in the sequence.  The initial prompt will come in
as an integer token value (range 0 - vocab_size-1).  You would one hot encode the initial prompt to be the 1st value in the sequence of the solo you are generating.

This function needs the following parameters as input:

- `model` A model already trained to do next-token prediction on the musical sequences, whose `predict()` method will return probability distribution of predicted next musical value.
- `sequence_length` The sequence length to generate.  Wouldn't necessarily have to be the same 30 sequence length we trained with
- `vocab_size` The vocabulary size, this does need to be equal to the vocabulary size of the model that was created
- `prompt` An integer value to use for the initial prompt solo / sentence.  This will need to be one-hot encoded before doing first prediction.  Should default to 0, the start token.

At the end we expect that you return an integer token encoded sequence, not one-hot.  So use `np.argmax` on the correct axis to encode back as integer indexes.  Also flatten
or squeeze this so you end up with a simple vector of shape `(sequence_length,)`.

In [ ]:
### TESTED function generate_jazz_solo()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_generate_jazz_solo'])

#solo = generate_jazz_solo(model, sequence_length, vocab_size, prompt=0, temperature=1.0)
#print(solo.shape)
#print(solo)
#print(type(solo))

**Example Expected Output**: You should get a vector of integer token indexes here.  The vector should be shaped `(30, )` and it should be a NumPy `ndarray` that is
returned. The first token index in the sequence should always be the prompt token, 0 in this example test.  However each time you run the solo sequence
should be different, as that is part of the point of a generative AI system here.

```
(30,)
[ 0 78 64 18 20 75 52 12  3 84 53 70 78 74 87 35 79 58 79 78 72 49 37 53
 45 53 13 78 40 75]
<class 'numpy.ndarray'>
```


**Task** Generate a solo using an initial `prompt` of 0 and a `temperature` of 1.0.
Once you have the solo, run the following cells.

The first cell will generate a `.midi` music file of the solo music values, and output it to a file named.  Play the file on your host system.  For windows the
basic media player should be able to open and play midi files.  Hopefully if you are on MacOS there default media player can play midi files as well (I haven't had
a chance to check).  

You can run the second cell to generate a wav file, which can be played in the notebook.  However the wave file generated is really bad and doesn't really
capture accurately what the solos sound like.  So if at all possible, to do the next task, play the midi file solos you generate using your
host system media player.

In [ ]:
# takes a generated solo, and creates a midi file for it, suitable for playing in a medial player that supports midi files
#out_stream = generate_music('../output/example1.midi', solo, vocab_dict, chords)

In [ ]:
#mid2wav('../output/example1.midi')
#IPython.display.Audio('../output/rendered.wav')

In [ ]:
# this might also work to play the midi output, but need to install pygame at least, maybe more
#from music21 import midi
#play = lambda x: midi.realtime.StreamPlayer(x).play()
#play(out_stream)

**Task** Hopefully you can find a media player to listen to your jazz solo midi files.  What works well for a generative AI system
is a bit subjective.  Demonstrate generating a solo with some different temperature ranges and listen to them.  You might also want to try with some different
initial `prompt` musical values.

**Hint**: might want to be a bit systematic, e.g. some loops to try several temperatures and prompts out to different file names so can compare all together.

In [ ]:
# show some examples of stochastic sampling here

**Task**:  You don't have to report too much here.  But I would be interested if you felt there were any good values of temperature or initial prompts.  Did
anything seem to strike you as more interesting than some other settings? Did you see any difference using a high temperature from using a low (or 0) temperature?

**Observations here**:

# Task 3: Build a Transformer based Sequence-to-Sequence Next-Token Model

You won't necessarily get more interesting or better results using a Transformer based model for your generator, but let's do a new model using
a `TransformerDecoder` layer.  As we mentioned in our course materials, the [Keras Hub](https://keras.io/keras_hub/)
package does have implementations of a `TransformerDecoder`available in it.  If you are using the
class development environment, the `keras_hub` library has been installed in the environment and the two layers you need have been imported.
If you are not using our class environment, you may need to install using `conda` or `pip` or whatever python package management system you are using.
We won't use any positional embedding for this model, a bit of testing shows it doesn't work well here.

**Task** Define and implement a function named `transformer_music_generator_model()`  This model creating function should be similar to your
previous one, and to the Listing 12.6 from our text.  

This model should take the following inputs again:

- `vocab_size` The size of the vocabulary being used in the one-hot encoded input and target sequences.
- `num_dim` The number of hidden dimensions to use for the Transformer layer.  
- `num_heads` The number of parallel transformer heads to use in the decoder.
   
You should use the Keras functional API to create a model and return it. 


In [ ]:
### TESTED function transformer_music_generator_model()
# uncomment when ready to run the unit tests for function
#run_unittests(['test_transformer_music_generator_model'])

num_dim = 32
num_heads = 2
#model = transformer_music_generator_model(vocab_size, num_dim, num_heads)
#model.summary()
#keras.utils.plot_model(model, show_shapes=True, show_layer_names=True)

**Expected Output**: You should get the following summary and network graph for your model using a transformer and position embedding
layer.

```
Model: "transformer_music_generator_model"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ inputs (InputLayer)             │ (None, None, 90)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder             │ (None, None, 90)       │        39,002 │
│ (TransformerDecoder)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ outputs (Dense)                 │ (None, None, 90)       │         8,190 │
└─────────────────────────────────┴────────────────────────┴───────────────┘
 Total params: 47,192 (184.34 KB)
 Trainable params: 47,192 (184.34 KB)
 Non-trainable params: 0 (0.00 B)
```

![Expected Transformer music generate network graph](../figures/expected_transformer_music_generator_model.png)

**Task**: In the following cell(s) train your model on the input and offset output music sequences.  Use the following guidance

- Use a basic `rmsprop` optimizer and report `accuracy` metric when fitting the model.  Use the correct loss for doing
  a multiclass classification.  In this case the `X` inputs and the `Y` output targets both have one-hot encoded representations.
- Train your model for 200 epochs (it will be fast).  Use a batch size of 10.  You don't need any validation data here for training a model for generating music.
- Plot the learning curve history, `plot_history()` is available to use.

In [ ]:
# compile your model for multi class classification using rmsprop and reporting on accuracy

In [ ]:
# fit your model for 200 epochs with a batch size of 10

In [ ]:
# show the learning curves of the training loss and accuracy

**Expected Results**: Usually you should find that the `TransformerDecoder` plateaus at around the same level of 92% accuracy, though it
usually gets there much faster.

**Task**: Demonstrate again generating some jazz solos with some different temperatures.  The function
to generate solos should work the same way with your new model.  I suspect you might not hear any difference
in the models here.

In [ ]:
# demonstrate stochastic sampling for transfer model

# Optional Task: Break 92%

Both simple one layer sequence-to-sequence models will probably hit about 92% but no better on your test accuracy.  It would be
interesting to see if adding more power or additional recurrent layers might make it possible to train a model that gets
higher accuracy.  And if so, what if any effect this might have on the quality of the jazz solos you can produce.

# Summary

Congratulations on completing this assignment. You should now have a bit of a better idea of the basics of how a generative AI system works in practice
to learn to predict the next token.  And how stochastic sampling can be implemented to help generate more interesting sequences.

<font color='blue'>
    
**What to remember from this assignment:**

- A sequence model can be used to generate musical values, which are then post-processed into midi music. 
- You can use a fairly similar model for tasks ranging from generating text, to music, to speech, with the only major difference being the input fed to the model.  
- In Keras, a sequence-to-sequence next-token prediction model is trained by feeding in inputs and the same sequence offset by 1 (or the desired) number of time steps.
- One-hot encoding can be used for basic sequence vocabularies, but you might want to consider embedding layers if the vocabulary is any more complex.